# Page Index- Vectorless RAG

Reasoning based- RAG, no chunking and vectordb

## Data Ingestion Pipeline
<b>Traditional RAG</b>: Data Ingestion -> chunking -> embedding -> store vectorDB

<b>Vectorless RAG</b>: Data Ingestion -> Looks at Table of Content ->LLM Tree Builder -> JSON Tree Index

#### What Happens if Table of Content is absent?

We will give any raw pdf document -> there will be document detection.
- if the TOC is present then -> parse the existing TOC -> Split sectionwise -> LLM summarizes each section.
- if TOC is not present then -> LLM reads every Page and analyzes the side headings and structure -> then it does sectionWise splitting -> LLM summarizes each section.

## Retrieval Pipeline
<b>Traditional RAG</b>: User Query -> Embeddings -> Searches similarity from VectorDB -> returns the context to LLM -> LLM returns the response.

<b>VectorLess RAG (Page Index)</b>: User Query -> LLM Tree Search(Reasons over tree tructure) -> reason and select the node -> extract section content -> LLM checks whether it is sufficient to answer or not.

- if that context is sufficient then, answer is generated by LLM and returned,
- else again the search and find node happens and this process repeats

In [ ]:
!pip install -U pageindex  python-dotenv

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for prettyprint: filename=prettyprint-0.1.5-py3-none-any.whl size=3086 sha256=770ad643aa8a1721bc1fc165a89e6f4408fa3601dbd23dba162440a65896f62d
  Stored in directory: c:\users\sanim\appdata\local\pip\cache\wheels\83\d6\f5\d5a2205656722a825a34bd2c92dcdf1c41af7ee64367ffce6d
Successfully built prettyprint


  DEPRECATION: Building 'prettyprint' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'prettyprint'. Discussion can be found at https://github.com/pypa/pip/issues/6334

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\sanim\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [47]:
import os
from dotenv import load_dotenv
load_dotenv()
GOOGLE_API_KEY=os.getenv('GOOGLE_API_KEY')
PAGE_INDEX_API_KEY=os.getenv('PAGE_INDEX_API_KEY')
GROQ_API_KEY=os.getenv('GROQ_API_KEY')

In [49]:
from pageindex import PageIndexClient
from groq import Groq
from google import genai

pi_client=PageIndexClient(api_key=PAGE_INDEX_API_KEY)
gq_client=Groq(api_key=GROQ_API_KEY)
go_client=genai.Client(api_key=GOOGLE_API_KEY)

In [50]:
PDF_PATH='./pdffiles/Attention is what all you need.pdf'
print('uploading document')
#submit_document- automatically the tree is built and also ocr is done for the pdf
result=pi_client.submit_document(PDF_PATH)
doc_id=result["doc_id"]
result # this result will consist a doc_id of the document we have just uploaded


uploading document


{'doc_id': 'pi-cmq7pa8lp001301qomsi9zetl'}

In [51]:
print('building Tree')
while True:
    #get_document returns dict containing id,metadata,description,status,pagenumbers,createdAt
    status_result=pi_client.get_document(doc_id=doc_id) 
    status=status_result.get("status")
    print(status)
    if(status=='completed'):
        print('tree index ready')
        break
    else:
        print('failed to build the tree')
        break

building Tree
completed
tree index ready


In [52]:
import json
tree_result=pi_client.get_tree(doc_id,node_summary=True)
pageindex_tree=tree_result['result']
# tree_result
print(json.dumps(pageindex_tree))

pageindex_tree

[{"title": "Attention Is All You Need", "node_id": "0000", "page_index": 1, "prefix_summary": "# Attention Is All You Need\n\n|  Ashish Vaswani* Google Brain avaswani@google.com | Noam Shazeer* Google Brain noam@google.com | Niki Parmar* Google Research nikip@google.com | Jakob Uszkoreit* Google Research usz@google.com  |\n| --- | --- | --- | --- |\n|  Llion Jones* Google Research llion@google.com | Aidan N. Gomez*\u2020 University of Toronto aidan@cs.toronto.edu | \u0141ukasz Kaiser* Google Brain lukaszkaiser@google.com  |   |\n\nIllia Polosukhin*\u2020\nillia.polosukhin@gmail.com\n", "text": "# Attention Is All You Need\n\n|  Ashish Vaswani* Google Brain avaswani@google.com | Noam Shazeer* Google Brain noam@google.com | Niki Parmar* Google Research nikip@google.com | Jakob Uszkoreit* Google Research usz@google.com  |\n| --- | --- | --- | --- |\n|  Llion Jones* Google Research llion@google.com | Aidan N. Gomez*\u2020 University of Toronto aidan@cs.toronto.edu | \u0141ukasz Kaiser* Goo

[{'title': 'Attention Is All You Need',
  'node_id': '0000',
  'page_index': 1,
  'prefix_summary': '# Attention Is All You Need\n\n|  Ashish Vaswani* Google Brain avaswani@google.com | Noam Shazeer* Google Brain noam@google.com | Niki Parmar* Google Research nikip@google.com | Jakob Uszkoreit* Google Research usz@google.com  |\n| --- | --- | --- | --- |\n|  Llion Jones* Google Research llion@google.com | Aidan N. Gomez*† University of Toronto aidan@cs.toronto.edu | Łukasz Kaiser* Google Brain lukaszkaiser@google.com  |   |\n\nIllia Polosukhin*†\nillia.polosukhin@gmail.com\n',
  'text': '# Attention Is All You Need\n\n|  Ashish Vaswani* Google Brain avaswani@google.com | Noam Shazeer* Google Brain noam@google.com | Niki Parmar* Google Research nikip@google.com | Jakob Uszkoreit* Google Research usz@google.com  |\n| --- | --- | --- | --- |\n|  Llion Jones* Google Research llion@google.com | Aidan N. Gomez*† University of Toronto aidan@cs.toronto.edu | Łukasz Kaiser* Google Brain lukaszk

In [53]:
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix=' '*indent+("└─ " if indent>0 else "")
        page=node.get('page_index',"?")
        print(f'{prefix}[{node['node_id']}] {node['title']} (p.{page})')
        if node.get("nodes"):
            print_tree(node["nodes"],indent+1)
print('Full Document Structure')
print_tree(pageindex_tree)

Full Document Structure
[0000] Attention Is All You Need (p.1)
 └─ [0001] Abstract (p.1)
 └─ [0002] 2 Background (p.2)
 └─ [0003] 3 Model Architecture (p.2)
  └─ [0004] 3.1 Encoder and Decoder Stacks (p.3)
  └─ [0005] 3.2 Attention (p.3)
   └─ [0006] 3.2.1 Scaled Dot-Product Attention (p.4)
   └─ [0007] 3.2.2 Multi-Head Attention (p.4)
   └─ [0008] 3.2.3 Applications of Attention in our Model (p.5)
  └─ [0009] 3.3 Position-wise Feed-Forward Networks (p.5)
  └─ [0010] 3.4 Embeddings and Softmax (p.5)
  └─ [0011] 3.5 Positional Encoding (p.6)
 └─ [0012] 4 Why Self-Attention (p.6)
 └─ [0013] 5 Training (p.7)
 └─ [0014] 6 Results (p.8)
  └─ [0015] 6.1 Machine Translation (p.8)
  └─ [0016] 6.2 Model Variations (p.8)
  └─ [0017] 6.3 English Constituency Parsing (p.9)
 └─ [0018] 7 Conclusion (p.10)
 └─ [0019] References (p.10)
 └─ [0020] Attention Visualizations (p.13)


In [54]:
def count_nodes(nodes):
    total_nodes=len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total_nodes+=count_nodes(n["nodes"])
    return total_nodes
print(f"The total number of nodes {count_nodes(pageindex_tree)}")

The total number of nodes 21


In [80]:
def llm_tree_search(query:str, tree:list, model:str='gemini-2.5-flash-lite')->dict:
    def compress(nodes):
        out=[]
        for n in nodes:
            entry={
                "node_id":n["node_id"],
                "title":n["title"],
                "page":n.get("page_index","?"),
                "summary":n.get("text")
            }
            if n.get("nodes"):
                entry["children"]=compress(n["nodes"])
            out.append(entry)
        return out
    compressed_tree=compress(tree)
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response=go_client.models.generate_content(
        model=model,
        contents=prompt
    )
    return response.candidates[0].content.parts[0].text


In [81]:
query="According to Section 3.2.2, why do the authors use multiple attention heads instead of a single attention head with the full dimensionality $d_{\text{model}}$?"
result=llm_tree_search(query,pageindex_tree)


In [ ]:
print(result)
type(result)

```json
{
  "thinking": "The user is asking a specific question about why multiple attention heads are used instead of a single one, according to Section 3.2.2. I need to locate Section 3.2.2 in the document tree and extract the relevant information that explains the benefit of multi-head attention over single-head attention with full dimensionality.",
  "node_list": ["0007"]
}
```


str